In [1]:


text = """
LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。

很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。

在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。

向量数据库（Vector Store）本质上保存的是 embedding 向量，而不是原始文本本身。用户查询时，系统会把 query 转换成 embedding，然后进行 semantic similarity search。

有些系统使用 cosine similarity，也有些使用 ANN（Approximate Nearest Neighbor）算法，比如 HNSW 或 IVF。

MultiQueryRetriever 是 LangChain 中一种提升召回率（recall）的技术。它会让 LLM 自动生成多个不同表达的查询语句，例如：

- “如何提高 RAG 检索效果”
- “怎样优化知识库召回”
- “improve retrieval quality in vector search”
- “增强 embedding 检索命中率的方法”

这些 query 会分别执行向量检索，然后对结果进行 merge 和 deduplicate。

这种方法特别适用于：
1. 用户问题表达模糊
2. chunk 中关键词不一致
3. 存在大量同义词
4. embedding 模型语义能力有限

例如，一篇文档中写的是“高并发缓存系统”，但用户问的是“redis scalability architecture”，单一 query 可能无法命中。

同样，“authentication” 和 “登录鉴权” 在某些 embedding 模型中距离并不近。

因此，多查询（multi query）能够从不同角度改写用户问题，提高 retrieval recall。

除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hypothetical Document Embeddings）
- Parent Document Retriever
- Contextual Compression Retriever
- Reranker
- BM25 + Vector Hybrid Search
- Self Query Retriever

在生产环境中，很多团队会结合 rerank model 与 hybrid retrieval 一起使用，以降低 hallucination 并提升 answer grounding。

LangGraph 是 LangChain 生态中的一个状态机工作流框架，用于构建 multi-agent systems、long-running workflow 和复杂推理链。

有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orchestration engine
- DeepAgents 更像高级 agent runtime

在 AI Coding 场景中，RAG 常被用于：
- 代码库问答
- 文档检索
- API 搜索
- 企业知识库
- Wiki assistant

一个典型 pipeline 通常包括：

Document Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator

如果 chunk size 太小，可能导致 context fragmentation；
如果 chunk 太大，又可能降低 embedding precision。

因此，chunk overlap、chunking strategy、metadata filtering 都会影响最终 retrieval performance。"""

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)
chunks = splitter.split_text(text)
print(len(chunks))

6


In [18]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

model = OllamaEmbeddings(
    model="qwen3-embedding:8b",
    base_url="http://192.168.31.198:11434",
)
vector_store = Chroma(
    collection_name="c3-3-multiquery",  # 给向量库起一个名字
    embedding_function=model,
    persist_directory="./chroma_db" # 指定数据存放的文件夹
)


In [19]:
vector_store.add_texts(chunks)

['7d80d8a7-25c5-4087-9c56-4f1a89c56058',
 '7567e751-ecef-4dce-913b-ceab95b666c4',
 '1807e809-be4b-4e6d-9216-e898f26250c6',
 '0453844c-b95d-4a4b-9b67-f167cfa7e964',
 'c4ed4c81-2fcb-40ff-8984-7754cd074347',
 '55f26ea9-bccf-49ed-b6a6-0f0daaba31f9']

## langchain 1.* 已经弃用MultiQueryRetriever
那就自己手动实现一个

In [58]:
from pydantic import BaseModel, Field

llm = ChatOllama(
    base_url="http://192.168.31.198:11434",
    model="qwen3:8b"
)
class Questions(BaseModel):
    """生成的三个问题"""
    question1: str = Field(description="生成的第1个问题")
    question2: str = Field(description="生成的第2个问题")
    question3: str = Field(description="生成的第3个问题")

query = "什么是langchain"
# prompt = f"""
# 原始问题是{query}, 帮我生成语义相近的三个问题
# """
prompt = f"""
你是一个 AI 语言模型助手，你的任务是生成给定用户问题的三个不同版本，以从在向量数据库中检索相关文档。
通过提供用户问题的多个视角，你的目标是帮助用户克服基于距离的相似性搜索的一些限制。
原始问题是：{query},
"""
llm_with_structured_output = llm.with_structured_output(
    Questions
)
queries = llm_with_structured_output.invoke(prompt)
print(queries)

question1='LangChain的核心功能和应用场景是什么？' question2='LangChain与其他对话系统（如Rasa、Dialogflow）相比有哪些优势和局限？' question3='如何在实际项目中使用LangChain进行对话系统开发？'


## 检索

### 返回结果去重

In [59]:
search_result = []
seen_ids = set()

for q in queries:
    docs = vector_store.similarity_search_with_relevance_scores(q[1])

    for doc, score in docs:
        if doc.id not in seen_ids:
            seen_ids.add(doc.id)
            search_result.append((doc, score))

for doc, score in search_result:
    print("="*20, score, "="*20)
    print(doc.page_content)

==================== 0.7505083163450335 ====================
LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。

很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。

在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。
==================== 0.5173194130157173 ====================
有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orchestration engine
- DeepAgents 更像高级 agent runtime

在 AI Coding 场景中，RAG 常被用于：
- 代码库问答
- 文档检索
- API 搜索
- 企业知识库
- Wiki assistant

一个典型 pipeline 通常包括：

Document Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator

如果 chunk size 太小，可能导致 context fragmentation；
如果 chunk 太大，又可能降低 embedding precision。
==================== 0.40801017055671274 ====================
除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hypothetical Document Embeddings）
- Parent Document Retriever
- Contextual Compress

## 注意：部分向量数据库返回的 doc.id 可能是 None。此时可以按照内容去重：

In [ ]:
search_result = []
seen_contents = set()

for q in queries:
    docs = vector_store.similarity_search_with_relevance_scores(q[1])

    for doc, score in docs:
        if doc.page_content not in seen_contents:
            seen_contents.add(doc.page_content)
            search_result.append((doc, score))

for doc, score in search_result:
    print("="*20, score, "="*20)
    print(doc.page_content)

## 拼接问题和检索结果成一个 prompt

In [60]:

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
reference_text = ""
for doc, score in search_result:
    reference_text += doc.page_content

prompt_all = f"""
用户的问题是:{query}
参考文本是:{reference_text}
"""
print(prompt_all)
chat_model = init_chat_model(
        base_url="http://192.168.31.198:11434",
        model="qwen3:8b",
        model_provider="ollama"
)
agent = create_agent(
    model=chat_model,
    system_prompt="你是一个专业的问答助手，请以用户给定的资料回答内容，如果用户给定的参考内容中不足以回答问题，则回答不知道。"
)
answer = agent.invoke({"messages": [
        {"role": "user", "content": prompt_all}
    ]})
print(answer)


用户的问题是:什么是langchain
参考文本是:LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。

很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。

在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orchestration engine
- DeepAgents 更像高级 agent runtime

在 AI Coding 场景中，RAG 常被用于：
- 代码库问答
- 文档检索
- API 搜索
- 企业知识库
- Wiki assistant

一个典型 pipeline 通常包括：

Document Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator

如果 chunk size 太小，可能导致 context fragmentation；
如果 chunk 太大，又可能降低 embedding precision。除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hypothetical Document Embeddings）
- Parent Document Retriever
- Contextual Compression Retriever
- Reranker
- BM25 + Vector Hybrid Search
- Self Query Retriever

在生产环境中，很多团队会结合 rerank model 与 hybrid retrieval 一起使用，以降低 hallucination 并提升 answer

In [47]:
print(answer["messages"][-1].content)

LangChain 是一个用于构建大语言模型（LLM）应用的框架，主要功能包括开发 AI Agent（智能代理）、检索增强生成（RAG）系统以及工作流编排。它被开发者比喻为“大模型时代的 Spring Framework”，因为其提供了模块化能力，如提示词管理（prompt management）、记忆（memory）、工具调用（tool calling）、检索器（retriever）和代理工作流编排（agent orchestration）等。

在 RAG 场景中，LangChain 通过将文本分块（chunk）存储到向量数据库（如 Chroma、FAISS 等）实现语义检索，结合 LLM 生成更准确的回答。其典型流程包括文档加载、文本分割、嵌入向量化、向量存储、检索和最终回答生成。同时，LangChain 支持多种优化技术（如 MultiQueryRetriever、HyDE、reranker 等）以提升检索效果和减少幻觉问题。

此外，LangChain 生态中还包含 LangGraph（用于构建多代理系统和复杂流程）等工具，但 LangChain 本身更偏向应用层抽象，而 LangGraph 更侧重工作流引擎。
